# Credit Scoring and PD Calibration

**Purpose:** Show how the `finstack_quant.core.credit` bindings support both classic bankruptcy-score style models and PiT/TtC PD transformations.

**Prerequisites:** Basic credit ratios and familiarity with point-in-time versus through-the-cycle default probabilities.

**In this notebook:** We compute Altman, Ohlson, and Zmijewski scores, then connect them to macro-conditioned PD calibration using the Merton-Vasicek mapping.


## Concept

This notebook combines two adjacent workflows:

1. **Cross-sectional scoring** from a company's balance-sheet and income-statement ratios.
2. **PD calibration** that converts long-run TtC views into stressed or benign PiT views.

Together they cover both the qualitative screening question and the quantitative default-probability question.


In [1]:
import math

import sys
sys.path.insert(0, "../..")

from _shared import banner
from finstack_quant.core.credit import pd as credit_pd
from finstack_quant.core.credit import scoring

# The scoring bindings take normalized, units-agnostic ratios, so the ratio prep
# below is call-site input assembly rather than modeling logic.
total_assets = 1_000.0

total_liabilities = 600.0
current_assets = 400.0
current_liabilities = 250.0
working_capital = current_assets - current_liabilities
retained_earnings = 200.0
ebit = 120.0
sales = 1_400.0
market_equity = 900.0
book_equity = total_assets - total_liabilities
net_income = 60.0
funds_from_operations = 140.0

wc_ta = working_capital / total_assets
re_ta = retained_earnings / total_assets
ebit_ta = ebit / total_assets
sales_ta = sales / total_assets
mkt_eq_liab = market_equity / total_liabilities
bk_eq_liab = book_equity / total_liabilities
tl_ta = total_liabilities / total_assets
cl_ca = current_liabilities / current_assets
ca_cl = current_assets / current_liabilities
ni_ta = net_income / total_assets
ffo_tl = funds_from_operations / total_liabilities


## Scores, zones, and macro conditioning

The next cell prints the sample company inputs, compares the standard score families, and then runs a PiT/TtC round-trip. Altman models provide canonical scores and zones but no native probability; the legacy score-to-PD mapping is available only through the explicitly selected, uncalibrated `HEURISTIC_V1` house heuristic.


In [2]:
banner("Sample company financials")
print(f"Total assets         : {total_assets:,.0f}")
print(f"Total liabilities    : {total_liabilities:,.0f}")
print(f"Working capital      : {working_capital:,.0f}")
print(f"Retained earnings    : {retained_earnings:,.0f}")
print(f"EBIT                 : {ebit:,.0f}")
print(f"Sales                : {sales:,.0f}")
print(f"Market cap           : {market_equity:,.0f}")

banner("Altman / Ohlson / Zmijewski")
z_score, z_zone, z_pd = scoring.altman_z_score(wc_ta, re_ta, ebit_ta, mkt_eq_liab, sales_ta)
zp_score, zp_zone, zp_pd = scoring.altman_z_prime(wc_ta, re_ta, ebit_ta, bk_eq_liab, sales_ta)
zdp_score, zdp_zone, zdp_pd = scoring.altman_z_double_prime(wc_ta, re_ta, ebit_ta, bk_eq_liab)
_, _, z_heuristic_pd = scoring.altman_z_score(
    wc_ta,
    re_ta,
    ebit_ta,
    mkt_eq_liab,
    sales_ta,
    scoring.AltmanPdCalibration.HEURISTIC_V1,
)
o_score, o_zone, o_pd = scoring.ohlson_o_score(
    math.log(total_assets),
    tl_ta,
    wc_ta,
    cl_ca,
    1.0 if total_liabilities > total_assets else 0.0,
    ni_ta,
    ffo_tl,
    0.0,
    0.05,
)
y_score, y_zone, y_pd = scoring.zmijewski_score(ni_ta, tl_ta, ca_cl)

print(f"Altman Z        : {z_score:.3f}  zone={z_zone:<8}  native PD={z_pd}")
print(f"Altman Z'       : {zp_score:.3f}  zone={zp_zone:<8}  native PD={zp_pd}")
print(f"Altman Z''      : {zdp_score:.3f}  zone={zdp_zone:<8}  native PD={zdp_pd}")
print(f"Altman HeuristicV1 (uncalibrated): {z_heuristic_pd:.2%}")
print(f"Ohlson O-Score  : {o_score:.3f}  zone={o_zone:<8}  PD={o_pd:.2%}")
print(f"Zmijewski Y     : {y_score:.3f}  zone={y_zone:<8}  PD={y_pd:.2%}")

banner("PiT / TtC conversion")
rho = 0.15
z_down = -1.5
z_bene = 1.0
ttc_pd = 0.020
pit_down = credit_pd.ttc_to_pit(ttc_pd, rho, z_down)
pit_bene = credit_pd.ttc_to_pit(ttc_pd, rho, z_bene)
rt_ttc = credit_pd.pit_to_ttc(pit_down, rho, z_down)
rt_pit = credit_pd.ttc_to_pit(rt_ttc, rho, z_down)

print(f"Input TtC PD        : {ttc_pd:.2%}")
print(f"Downturn PiT PD     : {pit_down:.2%}")
print(f"Benign PiT PD       : {pit_bene:.2%}")
print(f"Round-trip TtC      : {rt_ttc:.2%}")
print(f"Round-trip PiT      : {rt_pit:.2%}")

history = [0.015, 0.022, 0.018, 0.030, 0.012, 0.020, 0.025]
ct = credit_pd.central_tendency(history)
print(f"Central tendency    : {ct:.4%}")



Sample company financials
Total assets         : 1,000
Total liabilities    : 600
Working capital      : 150
Retained earnings    : 200
EBIT                 : 120
Sales                : 1,400
Market cap           : 900

Altman / Ohlson / Zmijewski
Altman Z        : 3.156  zone=safe      native PD=None
Altman Z'       : 2.327  zone=grey      native PD=None
Altman Z''      : 3.142  zone=safe      native PD=None
Altman HeuristicV1 (uncalibrated): 0.92%
Ohlson O-Score  : -1.276  zone=distress  PD=21.82%
Zmijewski Y     : -1.193  zone=grey      PD=11.64%

PiT / TtC conversion
Input TtC PD        : 2.00%
Downturn PiT PD     : 5.51%
Benign PiT PD       : 0.41%
Round-trip TtC      : 2.00%
Round-trip PiT      : 5.51%
Central tendency    : 2.0286%


## Scorecard extension note

This notebook demonstrates core `rating_scales` + PD term-structure usage. The `statements_analytics` credit scorecard extension (`ScorecardMetric`/`Config`/`Report`, `CreditScorecardExtension`, `validate_scorecard_config`) provides a separate weighted custom-metric scorecard builder; it is not exercised here to keep focus on the canonical rating / PD workflow.


In [3]:
import json as _json

from finstack_quant.statements_analytics import (
    ScorecardConfig,
    ScorecardMetric,
    ScorecardReport,
    CreditScorecardExtension,
    validate_scorecard_config,
)

# A scorecard maps financial-ratio formulas + rating thresholds to an overall grade.
leverage = ScorecardMetric(
    name="leverage",
    formula="debt / ebitda",
    weight=0.6,
    thresholds_json=_json.dumps({"A": [0.0, 2.0], "B": [2.0, 4.0], "C": [4.0, 99.0]}),
)
coverage = ScorecardMetric(
    name="coverage",
    formula="ebitda / interest",
    weight=0.4,
    thresholds_json=_json.dumps({"A": [6.0, 99.0], "B": [3.0, 6.0], "C": [0.0, 3.0]}),
)
config = ScorecardConfig(rating_scale="S&P", metrics=[leverage, coverage], period="2024Q4")
validate_scorecard_config(config)
print(f"ScorecardConfig: scale={config.rating_scale} metrics={[m.name for m in config.metrics]}")

# The extension is configured fluently and executed against a model (see statement_analytics for run_* APIs).
ext = CreditScorecardExtension().with_config(config)
print("CreditScorecardExtension configured:", ext.config is not None)
print("Produces a", ScorecardReport.__name__, "when executed against a model + results.")

ScorecardConfig: scale=S&P metrics=['leverage', 'coverage']
CreditScorecardExtension configured: True
Produces a ScorecardReport when executed against a model + results.


## Takeaways

- The scoring functions give a quick, explainable view of financial distress risk.
- The `pd` helpers let you separate **structural credit quality** from **cycle state**.
- In real use, these are complementary: scorecards can inform screening, while PiT/TtC conversions can inform provisioning, stress tests, or credit portfolio scenarios.


In [4]:
{
    "altman_zone": z_zone,
    "ohlson_zone": o_zone,
    "ttc_pd": round(ttc_pd, 4),
    "pit_down": round(pit_down, 4),
    "pit_bene": round(pit_bene, 4),
    "central_tendency": round(ct, 4),
}


{'altman_zone': 'safe',
 'ohlson_zone': 'distress',
 'ttc_pd': 0.02,
 'pit_down': 0.0551,
 'pit_bene': 0.0041,
 'central_tendency': 0.0203}